In [6]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


C:\Users\Dell\AppData\Local\Temp\ipykernel_16624\2389807203.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [7]:
load_dotenv()

True

In [8]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environment variable loaded :)")

Environment variable loaded :)


In [29]:
#Loading our data file
DATA_DIR = os.path.join(os.getcwd(), "data","hr_policy.txt")

In [30]:
#data ingestion
loader = TextLoader(DATA_DIR,encoding="utf-8")
documents = loader.load()

print(documents)


[Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from th

In [31]:
print(documents[0].metadata)

{'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require writte

In [33]:
len(chunks)
print(chunks[1])

page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}


In [34]:
##Embed our data using Jina Embeddings
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("embeddings model loaded :)",embeddings_model.model_name)


embeddings model loaded :) jina-embeddings-v2-base-en


In [35]:
#Store data in vector database using Groq
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings_model)
print("vector store created :)",vector_store.index.ntotal)

vector store created :) 9


In [17]:
test_query = "How many sick leaves employees get?"
#similarity search
similar_docs = vector_store.similarity_search(test_query, k=3)
print("similar docs :)",similar_docs)
for i,match in enumerate(similar_docs):
    print(f"Match {i+1}:")
    print(f"Content: {match.page_content}")
    print(f"Metadata: {match.metadata}")
    print() 

similar docs :) [Document(id='d7e7b0eb-ce4a-43bd-9749-e87942e30e5a', metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(id='91564c64-1215-488f-b996-c4bc3c0d4603', metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'), Docume

In [36]:
#TOOL
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [37]:
#Data Retrieval using RAG
#LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.9,
)

llm.model_name

'openai/gpt-oss-120b'

In [38]:
test_response = llm.invoke("hey is learning Rag hard? answer in one line")
test_response.content

'Learning RAG can be challenging at first, but with good resources and practice it’s definitely manageable.'

In [39]:
#AI AGENT 3 LLM - brain , tool - super power , memory - no memory
from langchain.agents import create_agent


In [44]:
hr_assistant = create_agent(
    model=llm,
    tools = [search_hr_policy],
    system_prompt = """ You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """,
)
print("HR assistant is ready to answer your questions :)")
    

HR assistant is ready to answer your questions :)


In [45]:
def ask_hr_assistant(question:str) -> str:
    """send a question to the RAG and print the answer"""
    print("="*60)
    print("Question:",question)
    print("-" * 60)
    
    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    
    print("Answer:",answer)
    print("="*60)
    
    print()
    return answer

In [46]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "What is the leave policy?"
            }
        ]
    }
)
#answer = response

In [47]:
response

{'messages': [HumanMessage(content='What is the leave policy?', additional_kwargs={}, response_metadata={}, id='0e562682-09b4-498a-a70a-fd3d05370295'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer using search_hr_policy tool. The user asks "What is the leave policy?" Must query the HR policy doc. Use the tool.', 'tool_calls': [{'id': 'fc_7e171eee-08cb-4850-89ff-e8d94185af46', 'function': {'arguments': '{"question":"leave policy"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 206, 'total_tokens': 267, 'completion_time': 0.127810689, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.01018412, 'prompt_tokens_details': None, 'queue_time': 0.279408211, 'total_time': 0.137994809}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider':

In [42]:
#SYSTEM MESSAGE - HR ASSISNT
#HUMAN MESSAGE - TELL ME ABOUT POLICIES
#AI MESSAGE - HEY THESE ARE THEPLOICES
response["messages"][-2].content

'What is the leave policy?'

In [43]:
print(response["messages"][-1].content)

I don't know.
